In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
SYSTEM_PROMPT = """你是一个专业的代码审查专家。对提供的代码进行分析，输出一份结构化评审报告，涵盖以下五个维度：

1. **Bug 与正确性** — 逻辑错误、边界情况、异常处理
2. **安全问题** — 注入风险、密钥泄露、不安全的操作
3. **性能问题** — 效率低下、不必要的计算、内存占用
4. **代码风格** — PEP 8 违规、命名规范、可读性
5. **改进建议** — 重构建议、更好的设计模式

格式：使用 Markdown。给出总体质量评级:🟢 优秀 / 🟡 需改进 / 🔴 存在严重问题"""

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

import os

model = os.getenv("MODEL_NAME")

def review_code(code: str, language: str = "python") -> str:
    if model is None:
        raise ValueError("MODEL_NAME is not set")
    llm = ChatOpenAI(model=model, temperature=0, extra_body={"enable_thinking": False})
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Review 这份 {language} 代码:\n\n```{language}\n{code}\n```"),
    ]
    content = ''
    response = llm.stream(messages)
    for chunk in response:
        str = getattr(chunk, 'content')
        content += str
        print(str, end='', flush=False)

    return content

In [7]:
with open('/Users/linqibin/Documents/code/agents/practise_agents/apps/backend/agents/research_agent.ipynb') as f:
    code = f.read()
    review_code(code=code, language='python')

# 代码审查报告

## 总体质量评级: 🟡 需改进

这份代码实现了一个基于 LangGraph 的简单研究代理（Research Agent），能够搜索网络并生成报告。虽然核心逻辑可行，但在**状态管理、错误处理、资源效率和安全性**方面存在显著缺陷，直接用于生产环境可能会导致数据丢失、API 浪费或安全风险。

---

### 1. Bug 与正确性 (Bug & Correctness)

*   **严重：状态更新不完整 (`search_web`)**
    *   **问题**: `search_web` 函数只返回了 `{"search_results": results}`。在 LangGraph 中，如果节点不返回其他字段，这些字段在后续节点中可能会丢失或保持初始值（取决于 reducer 的实现）。虽然 `messages` 有 `add_messages` reducer，但 `query` 和 `report` 没有被显式传递或保留。如果图结构更复杂，这会导致 `synthesize_report` 无法访问 `state["query"]`。
    *   **修复**: 应该返回完整的状态字典，或者确保 StateGraph 的默认行为能保留未修改的字段（通常建议显式返回所有必要字段或使用 `return {**state, "search_results": results}` 模式，尽管 TypedDict + Annotated 有时允许部分更新，但显式更安全）。
    *   **代码片段**:
        ```python
        # 当前
        return {"search_results": results}
        
        # 建议
        return {**state, "search_results": results} 
        ```

*   **潜在 Bug: `synthesize_report` 中的消息追加**
    *   **问题**: `return {"report": response.content, "messages": [response]}`。这里将 LLM 的响应作为新消息列表传入。由于 `messages` 